# Walkthrough

The model in five steps, on the committed data in `data/processed`. Run `python -m warcost fetch` first if you want fresh data.

In [1]:
import pandas as pd
from warcost.analysis import human_cost, load_inputs, run_all, sensitivity
from warcost.config import load_config
from warcost.health import totals

pd.set_option("display.precision", 2)
cfg = load_config()
inputs = load_inputs(cfg)
results = run_all(cfg, inputs)

## 1. Who spends what

Budget shares of cash spending (imputed rent removed) from the CBS budget survey. The poorest fifth spends a larger share on home energy, a smaller one on motor fuel, and almost the same share on food.

In [2]:
from warcost.exposure import spending_weights
spending_weights(inputs["households"], cfg["cbs"]["survey_year"]).mul(100)

,food,home_energy,motor_fuel,core
group,,,,
Q1,13.51,5.76,1.78,78.95
Q2,14.32,5.52,2.71,77.45
Q3,15.76,5.54,3.82,74.88
Q4,15.82,5.32,3.83,75.03
Q5,15.12,4.51,3.66,76.71
all,15.10,5.23,3.33,76.34


## 2. The spending-to-income problem

The poorest fifth's survey spending exceeds its registered disposable income. Every result expressed as a share of income inherits this ratio.

In [3]:
survey = inputs["households"].loc["2020"].loc[["Q1", "Q2", "Q3", "Q4", "Q5"], "spending_eur"]
income_2020 = inputs["income"].loc[2020, "income_eur"]
(survey / income_2020).rename("spending / income, 2020")

group
Q1    1.34
Q2    0.96
Q3    0.85
Q4    0.76
Q5    0.57
Name: spending / income, 2020, dtype: float64

## 3. Losses per quintile

In [4]:
results["ukraine"].summary[["gross_eur", "compensation_eur", "net_eur", "gross_pct_spending", "gross_pct_income", "net_pct_income"]]

,gross_eur,compensation_eur,net_eur,gross_pct_spending,gross_pct_income,net_pct_income
quintile,,,,,,
Q1,1209.35,1810.0,-600.65,2.94,3.73,-1.85
Q2,1457.47,640.0,817.47,2.94,2.59,1.45
Q3,1762.10,380.0,1382.10,3.08,2.20,1.73
Q4,2207.27,380.0,1827.27,3.02,1.93,1.59
Q5,2725.63,380.0,2345.63,2.74,1.25,1.07


In [5]:
results["iran"].summary[["gross_eur", "compensation_eur", "net_eur", "gross_pct_spending", "gross_pct_income", "net_pct_income"]]

,gross_eur,compensation_eur,net_eur,gross_pct_spending,gross_pct_income,net_pct_income
quintile,,,,,,
Q1,24.22,10.0,14.22,0.19,0.24,0.14
Q2,47.89,25.0,22.89,0.32,0.27,0.13
Q3,81.09,40.0,41.09,0.47,0.32,0.16
Q4,103.74,55.0,48.74,0.47,0.28,0.13
Q5,134.82,60.0,74.82,0.45,0.19,0.11


## 4. How fragile is the headline?

Net loss as a share of income for the poorest and richest fifth under every combination of counterfactual, energy series and survey year.

In [6]:
sensitivity(cfg, inputs).query("episode == 'ukraine'").drop(columns="episode")

,counterfactual,energy_correction_weight,survey_year,q1_net_pct,q5_net_pct,q1_with_core_pct,q5_with_core_pct
0,target,0.0,2020,-2.74,0.82,2.44,2.63
2,target,0.0,2015,-2.89,0.73,2.30,2.69
4,target,0.5,2020,-1.85,1.07,3.33,2.88
6,target,0.5,2015,-1.95,1.00,3.24,2.95
8,target,1.0,2020,-0.96,1.33,4.22,3.13
10,target,1.0,2015,-1.01,1.26,4.18,3.21
12,pre_trend,0.0,2020,-4.60,0.21,0.11,1.85
14,pre_trend,0.0,2015,-4.98,0.02,-0.26,1.80
16,pre_trend,0.5,2020,-3.71,0.46,1.00,2.10
18,pre_trend,0.5,2015,-4.04,0.29,0.68,2.07


## 5. Life-years

Totals across quintiles. `lifetimes` is life-years divided by 82: an equivalence, not a count of deaths.

In [7]:
lives = human_cost(cfg, inputs, results)
pd.concat({episode: totals(frame) for episode, frame in lives.items()})

life_years  healthy_life_years  lifetimes  \
        scenario basis                                              
ukraine central  gross    15129.30            32038.51     184.50   
                 net       9258.50            19606.24     112.91   
        high     gross    30258.59            64077.02     369.01   
                 net      18517.00            39212.48     225.82   
        low      gross     6051.72            12815.40      73.80   
                 net       3703.40             7842.50      45.16   
iran    central  gross      507.21             1074.08       6.19   
                 net        259.49              549.51       3.16   
        high     gross     1014.41             2148.16      12.37   
                 net        518.98             1099.03       6.33   
        low      gross      202.88              429.63       2.47   
                 net        103.80              219.81       1.27   

                        days_per_person  
        scenario basis                   
ukraine central  gross         3.16e-01  
                 net           1.93e-01  
        high     gross         6.31e-01  
                 net           3.86e-01  
        low      gross         1.26e-01  
                 net           7.73e-02  
iran    central  gross         1.06e-02  
                 net           5.41e-03  
        high     gross         2.12e-02  
                 net           1.08e-02  
        low      gross         4.23e-03  
                 net           2.17e-03